In [134]:
# Imports
from gliner import GLiNER

import os
import sys
import dotenv
import re

import json
from pathlib import Path

from collections import defaultdict
from prettytable import PrettyTable

import torch
import accelerate

dotenv.load_dotenv()
ROOT_DIR = os.environ.get("ROOT_DIR")
sys.path.append(f"{ROOT_DIR}/scripts")

from evaluation import evaluate

In [135]:
test_before_2000 = json.load(open(f"{ROOT_DIR}/data/splits/test_before_2000.json", "r"))
test_after_2000 = json.load(open(f"{ROOT_DIR}/data/splits/test_after_2000.json", "r"))

# Raw model

## Inference 

In [136]:
def inference(model, data, tags, threshold):
    all_doc = []
    for doc in data:
        text = doc["texte"]
        entities = []
        # Diviser le texte en chunks si trop long
        max_length = 300  # caractères (pas tokens)
        chunks = []
        start_pos = 0
        while start_pos < len(text):
            # Trouver un point ou une virgule pour couper proprement
            end_pos = min(start_pos + max_length, len(text))
            if end_pos < len(text):
                # Chercher le dernier point/virgule/espace avant la limite
                last_punct = max(
                    text.rfind('. ', start_pos, end_pos),
                    text.rfind('! ', start_pos, end_pos),
                    text.rfind('? ', start_pos, end_pos),
                    text.rfind('\n', start_pos, end_pos)
                )
                if last_punct > start_pos:
                    end_pos = last_punct + 1
            chunk = text[start_pos:end_pos].strip()
            if chunk:
                chunks.append((chunk, start_pos))
            start_pos = end_pos
        
        # Prédire sur chaque chunk
        for chunk_text, chunk_offset in chunks:
            try:
                detected_entities = model.predict_entities(
                    text=chunk_text, 
                    labels=tags, 
                    threshold=threshold
                )
                for ent in detected_entities:
                    # Mapper les labels GLiNER aux tags NER standard
                    label_mapping = {
                        "person": "PER",
                        "location": "LOC",
                        "political party/political movement": "ORG",
                        "profession": "MISC"
                    }
                    
                    original_label = ent["label"].lower()
                    mapped_tag = label_mapping.get(original_label, "MISC")
                    
                    entities.append({
                        "texte": ent["text"],
                        "tag": mapped_tag,
                        "debut": ent["start"] + chunk_offset,
                        "fin": ent["end"] + chunk_offset
                    })
            except Exception as e:
                print(f"Erreur sur chunk {doc['id']}: {e}")
        
        all_doc.append({
            "id": doc["id"],
            "annee": doc["annee"],
            "predicted_entities": entities
        })
    return all_doc

In [150]:
model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1") #gliner_medium-v2.1 #urchade/gliner_base # urchade/gliner_multi_pii-v1
tags = ["person", "political party/political movement", "location", "profession"]

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9] #


/Users/wiamlachqer/Desktop/Named_Entity_Extraction_Archelec_Corpus/projet_archelec/.venv/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [151]:
for threshold in thresholds:
    print(f"----------------- Running inference with threshold {threshold} -----------------")

    prediction_before_2000 = inference(model, test_before_2000, tags=tags, threshold=threshold)
    prediction_after_2000 = inference(model, test_after_2000, tags=tags, threshold=threshold)

    metrics_before_2000 = evaluate(prediction_before_2000, test_before_2000)
    metrics_after_2000 = evaluate(prediction_after_2000, test_after_2000)

    # Chemin du fichier
    output_path_before_2000 = "../data/results/Gliner/raw/metrics_before_2000_{}.json".format(threshold)

    # Créer les répertoires s'ils n'existent pas
    os.makedirs(os.path.dirname(output_path_before_2000), exist_ok=True)

    # Sauvegarder le JSON
    with open(output_path_before_2000, 'w', encoding='utf-8') as f:
        json.dump(metrics_before_2000, f, indent=2, ensure_ascii=False)

    # Chemin du fichier
    output_path_after_2000 = "../data/results/Gliner/raw/metrics_after_2000_{}.json".format(threshold)

    # Créer les répertoires s'ils n'existent pas
    os.makedirs(os.path.dirname(output_path_after_2000), exist_ok=True)

    # Sauvegarder le JSON
    with open(output_path_after_2000, 'w', encoding='utf-8') as f:
        json.dump(metrics_after_2000, f, indent=2, ensure_ascii=False)

----------------- Running inference with threshold 0.1 -----------------
Global NER Performance (Exact Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.0843 |
|   Recall  | 0.4065 |
|  F1-Score | 0.1396 |
+-----------+--------+

Global NER Performance (Partial Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.1502 |
|   Recall  | 0.7243 |
|  F1-Score | 0.2488 |
+-----------+--------+

Performance by Tag — Exact Match
+------+-----------+--------+----------+---------+------------+
| Tag  | Precision | Recall | F1-Score | Support | Partial TP |
+------+-----------+--------+----------+---------+------------+
| LOC  |   0.0570  | 0.4691 |  0.1016  |    81   |     57     |
| MISC |   0.0761  | 0.4268 |  0.1292  |    82   |     14     |
| ORG  |   0.1222  | 0.2245 |  0.1583  |   196   |     36     |
| PER  |   0.0988  | 0.8261 |  0.1765  |    69   |     29     |
+------+-----------+--------+----------+------

Best threshold = 0,5

# Fine tuned model

In [139]:
def find_entity_offset(text, entity_text, hint):
    """
    Retrouve la vraie position de entity_text dans text.
    58.8% des offsets annotés (debut/fin) sont décalés dans ce corpus.
    Retourne l'occurrence la plus proche du hint.
    """
    text_lower = text.lower()
    ent_lower  = entity_text.strip().lower()
    if not ent_lower:
        return None, None
    occurrences = []
    start = 0
    while True:
        idx = text_lower.find(ent_lower, start)
        if idx == -1:
            break
        occurrences.append(idx)
        start = idx + 1
    if not occurrences:
        return None, None
    closest = min(occurrences, key=lambda x: abs(x - hint))
    return closest, closest + len(entity_text.strip())


# Tokenizer IDENTIQUE à celui utilisé par GLiNER en inférence (WhitespaceTokenSplitter)
# Source : gliner/data_processing/tokenizer.py  →  r"\w+(?:[-_]\w+)*|\S"
# ⚠  différent de r"\w+(?:[-']\w+)*|[^\w\s]"  :
#    - les apostrophes sont des tokens séparés  ("l'UMP" → ["l", "'", "UMP"])
#    - \S capture tout caractère non-blanc en 1 token (ponctuation incluse)
GLINER_TOKEN_RE = re.compile(r"\w+(?:[-_]\w+)*|\S")


def format_data(data, max_length=384, skip_empty=True):
    """
    Convertit les documents annotés au format GLiNER fine-tuning.

    tokenized_text : tokens produits par le même tokenizer que l'inférence GLiNER
    ner            : [[start, end, label], ...]
                     indices CHUNK-RELATIFS, end INCLUSIF  (convention GLiNER)

    Corrections :
      1. Vrais offsets recalculés  (58.8 % des debut/fin annotés sont décalés)
      2. Tokenizer = regex GLiNER  (apostrophes séparées, ponctuation séparée)
      3. end INCLUSIF              (prepare_span_idx de GLiNER est inclusif)
      4. Labels = strings identiques à l'inférence
    """

    TAG_TO_LABEL = {
        "PER":  "person",
        "LOC":  "location",
        "ORG":  "political party/political movement",
        "MISC": "profession",
    }

    formatted_data = []
    stats = {"docs": 0, "chunks": 0, "ner_total": 0,
             "ent_ok": 0, "ent_not_found": 0}

    for doc in data:
        text     = doc.get("texte", "").strip()
        entities = doc.get("entites", [])
        if not text:
            continue
        stats["docs"] += 1

        # ── 1. Tokenisation (même regex que GLiNER) ────────────────────────
        token_spans = [
            (m.group(), m.start(), m.end())
            for m in GLINER_TOKEN_RE.finditer(text)
        ]
        if not token_spans:
            continue

        # ── 2. Résolution des vrais offsets pour toutes les entités ────────
        resolved = []
        for ent in entities:
            cs, ce = find_entity_offset(text, ent["texte"], ent["debut"])
            if cs is None:
                stats["ent_not_found"] += 1
                continue
            label = TAG_TO_LABEL.get(ent["tag"], ent["tag"])
            resolved.append((cs, ce, label))
            stats["ent_ok"] += 1

        # ── 3. Découpage en chunks de max_length tokens ────────────────────
        for chunk_start_idx in range(0, len(token_spans), max_length):
            chunk = token_spans[chunk_start_idx : chunk_start_idx + max_length]
            if not chunk:
                continue

            chunk_tokens     = [t[0] for t in chunk]
            chunk_char_start = chunk[0][1]
            chunk_char_end   = chunk[-1][2]

            # ── 4. Convertir les offsets char → indices token (INCLUSIFS) ──
            ner_entities = []
            for ent_cs, ent_ce, label in resolved:

                if ent_ce <= chunk_char_start or ent_cs >= chunk_char_end:
                    continue

                token_start = None
                token_end   = None
                for i, (tok, cs, ce) in enumerate(chunk):
                    # premier token dont la fin dépasse le début de l'entité
                    if token_start is None and ce > ent_cs:
                        token_start = i
                    # dernier token dont le début est avant la fin de l'entité
                    # end INCLUSIF : on garde i (pas i+1)
                    if cs < ent_ce:
                        token_end = i

                # token_start <= token_end  (== pour entité mono-token)
                if (token_start is not None
                        and token_end is not None
                        and token_start <= token_end):
                    ner_entities.append([token_start, token_end, label])
                    stats["ner_total"] += 1

            if skip_empty and not ner_entities:
                continue

            formatted_data.append({
                "tokenized_text": chunk_tokens,
                "ner": ner_entities,
            })
            stats["chunks"] += 1

    # ── Affichage des statistiques ─────────────────────────────────────────
    print(f"Documents traités       : {stats['docs']}")
    print(f"Chunks générés          : {stats['chunks']}")
    print(f"Entités NER placées     : {stats['ner_total']}")
    print(f"Entités introuvables    : {stats['ent_not_found']}")
    coverage = 100 * stats['ent_ok'] / max(stats['ent_ok'] + stats['ent_not_found'], 1)
    print(f"Couverture des entités  : {coverage:.1f}%")
    print(f"NER / chunk (moy.)      : {stats['ner_total'] / max(stats['chunks'], 1):.2f}")

    return formatted_data


## Training

In [140]:
train_data = json.load(open(f"{ROOT_DIR}/data/splits/train.json", "r"))
train_dataset = format_data(train_data)

Documents traités       : 46
Chunks générés          : 127
Entités NER placées     : 1029
Entités introuvables    : 19
Couverture des entités  : 98.2%
NER / chunk (moy.)      : 8.10


In [141]:
os.environ["TOKENIZERS_PARALLELISM"] = "true"

In [142]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1") #urchade/gliner_small.   # urchade/gliner_medium-v2.1

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [143]:
# # Geler uniquement le backbone transformer (token_rep_layer)
# # model.model = SpanModel entier → il faut descendre à token_rep_layer
# for param in model.model.token_rep_layer.parameters():
#     param.requires_grad = False

# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
# print(f'Paramètres entraînables : {trainable:,}')  # ~12M couches hautes
# print(f'Paramètres gelés        : {frozen:,}')     # ~183M backbone DeBERTa``

# Geler tout le backbone sauf les 2 dernières couches
for name, param in model.model.token_rep_layer.named_parameters():
    param.requires_grad = False  # tout gelé par défaut

# Dégeler les 2 dernières couches encoder
for name, param in model.model.token_rep_layer.named_parameters():
    if any(f"encoder.layer.{i}" in name for i in [10, 11]):
        param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Paramètres entraînables : {trainable:,}")   # ~22M
print(f"Paramètres gelés        : {frozen:,}")      # ~173M

Paramètres entraînables : 25,200,128
Paramètres gelés        : 263,749,376


In [144]:
trainer = model.train_model(
    train_dataset=train_dataset,
    eval_dataset=None,
    output_dir="../models/Gliner/",
    learning_rate=1e-4,          # OK : backbone gelé, uniquement couches hautes
    weight_decay=0.01,
    others_lr=1e-4,
    others_weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=20,             # était 5/50
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    focal_loss_alpha=0.75,
    focal_loss_gamma=2,
    max_steps=800,               # ~6 epochs (66 steps/epoch × 6) — était 50/2000
    save_steps=100,              # ≤ max_steps — était 250
    save_total_limit=2,
    dataloader_num_workers=0,
    use_cpu=True,
    report_to="none",
    logging_steps=25,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 0}.


Step,Training Loss
25,16.885602
50,6.399531
75,6.743638
100,6.297986
125,5.779320
150,5.915092
175,4.708411
200,5.791742
225,4.496732
250,4.357408


In [148]:
trained_model = GLiNER.from_pretrained(
    os.path.abspath("../models/Gliner/checkpoint-800"),
    load_tokenizer=True,
    local_files_only=True
)
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9] #]. , 0.6, 0.7,     0.1, 0.2, 0.3, 0.4, 
tags = ["person", "political party/political movement", "location", "profession"]

In [149]:
for threshold in thresholds:
    print(f"----------------- Running inference with threshold {threshold} -----------------")

    prediction_before_2000 = inference(trained_model, test_before_2000, tags=tags, threshold=threshold)
    prediction_after_2000 = inference(trained_model, test_after_2000, tags=tags, threshold=threshold)

    metrics_before_2000 = evaluate(prediction_before_2000, test_before_2000)
    metrics_after_2000 = evaluate(prediction_after_2000, test_after_2000)

    # Chemin du fichier
    output_path_before_2000 = "../data/results/Gliner/trained/trained_metrics_before_2000_{}.json".format(threshold)

    # Créer les répertoires s'ils n'existent pas
    os.makedirs(os.path.dirname(output_path_before_2000), exist_ok=True)

    # Sauvegarder le JSON
    with open(output_path_before_2000, 'w', encoding='utf-8') as f:
        json.dump(metrics_before_2000, f, indent=2, ensure_ascii=False)

    # Chemin du fichier
    output_path_after_2000 = "../data/results/Gliner/trained/trained_metrics_after_2000_{}.json".format(threshold)

    # Créer les répertoires s'ils n'existent pas
    os.makedirs(os.path.dirname(output_path_after_2000), exist_ok=True)

    # Sauvegarder le JSON
    with open(output_path_after_2000, 'w', encoding='utf-8') as f:
        json.dump(metrics_after_2000, f, indent=2, ensure_ascii=False)

----------------- Running inference with threshold 0.1 -----------------
Global NER Performance (Exact Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.1130 |
|   Recall  | 0.5794 |
|  F1-Score | 0.1892 |
+-----------+--------+

Global NER Performance (Partial Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.1595 |
|   Recall  | 0.8178 |
|  F1-Score | 0.2670 |
+-----------+--------+

Performance by Tag — Exact Match
+------+-----------+--------+----------+---------+------------+
| Tag  | Precision | Recall | F1-Score | Support | Partial TP |
+------+-----------+--------+----------+---------+------------+
| LOC  |   0.1055  | 0.7531 |  0.1851  |    81   |     33     |
| MISC |   0.0557  | 0.5122 |  0.1005  |    82   |     23     |
| ORG  |   0.1268  | 0.4490 |  0.1978  |   196   |     32     |
| PER  |   0.3393  | 0.8261 |  0.4810  |    69   |     14     |
+------+-----------+--------+----------+------